# Notebook 07 · SQL serving público

Versión portfolio-ready que sustituye MySQL local por SQLite y usa únicamente datasets públicos.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

In [2]:
import json
import sqlite3
import re
from pathlib import Path

import pandas as pd
import yaml

SQL_DIR = ARTIFACTS_PUBLIC / "sql"
SQL_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = SQL_DIR / "public_serving.db"
CONTRACT_PATH = PROJECT_ROOT / "config" / "inference_input_contract.yaml"
VIEWS_SQL_PATH = PROJECT_ROOT / "src" / "sql" / "public_db_views_serving.sql"

contract = yaml.safe_load(CONTRACT_PATH.read_text(encoding="utf-8"))
print({"db_path": str(DB_PATH.relative_to(PROJECT_ROOT)), "contract_path": str(CONTRACT_PATH.relative_to(PROJECT_ROOT))})
contract

{'db_path': 'artifacts/public/sql/public_serving.db', 'contract_path': 'config/inference_input_contract.yaml'}


{'contract_id': 'inference_input_v1',
 'version': '1.0.0',
 'status': 'active',
 'owner': 'proyecto-ml-source',
 'description': 'Contrato mínimo de validación del input diario para inferencia de proveedor. Define grano, columnas obligatorias, tipos base, reglas de dominio y coherencia.\n',
 'grain': {'key_columns': ['event_id', 'proveedor_candidato']},
 'columns': {'date': ['fecha_evento'],
  'categorical_non_empty': ['event_id',
   'proveedor_candidato',
   'producto_canonico',
   'terminal_compra'],
  'numeric': ['coste_min_dia_proveedor',
   'rank_coste_dia_producto',
   'terminales_cubiertos',
   'observaciones_oferta',
   'candidatos_evento_count',
   'coste_min_evento',
   'coste_max_evento',
   'spread_coste_evento',
   'delta_vs_min_evento',
   'ratio_vs_min_evento',
   'litros_evento',
   'dia_semana',
   'mes',
   'fin_mes']},
 'domain': {'mes': {'min_inclusive': 1, 'max_inclusive': 12},
  'dia_semana': {'allowed': [0, 1, 2, 3, 4, 5, 6]},
  'fin_mes': {'allowed': [0, 1]},
  '

In [3]:
public_csv_paths = sorted((DATA_PUBLIC).rglob("*.csv"))
inventory_df = pd.DataFrame(
    [
        {
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "table_name": re.sub(r"[^a-z0-9_]+", "_", "public_" + path.stem.lower()),
            "rows_csv": max(sum(1 for _ in path.open("rb")) - 1, 0),
        }
        for path in public_csv_paths
    ]
)
inventory_df

,relative_path,table_name,rows_csv
0,data/public/dataset_modelo_proveedor_v1.csv,public_dataset_modelo_proveedor_v1,16315
1,data/public/dataset_modelo_proveedor_v2_candid...,public_dataset_modelo_proveedor_v2_candidates,155946
2,data/public/dataset_modelo_proveedor_v2_exclud...,public_dataset_modelo_proveedor_v2_excluded_ev...,911
3,data/public/dataset_modelo_proveedor_v3_contex...,public_dataset_modelo_proveedor_v3_context,155946
4,data/public/day041/dataset_modelo_v2_competiti...,public_dataset_modelo_v2_competition,155946
5,data/public/day041/dataset_modelo_v2_dispersio...,public_dataset_modelo_v2_dispersion,155946
6,data/public/day041/dataset_modelo_v2_source_qu...,public_dataset_modelo_v2_source_quality,155946
7,data/public/day041/dataset_modelo_v2_transport...,public_dataset_modelo_v2_transport_only,155946
8,data/public/day042/dataset_modelo_v2_dispersio...,public_dataset_modelo_v2_dispersion_plus_trans...,155946
9,data/public/day042/dataset_modelo_v2_transport...,public_dataset_modelo_v2_transport_rebuilt_only,155946


In [4]:
with sqlite3.connect(DB_PATH) as conn:
    load_rows = []
    for row in inventory_df.to_dict(orient="records"):
        source_path = PROJECT_ROOT / row["relative_path"]
        table_name = row["table_name"]
        df = pd.read_csv(source_path)
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        rows_loaded = pd.read_sql_query(f"SELECT COUNT(*) AS rows_loaded FROM {table_name}", conn).iloc[0]["rows_loaded"]
        load_rows.append(
            {
                "table_name": table_name,
                "rows_csv": int(row["rows_csv"]),
                "rows_loaded": int(rows_loaded),
                "load_ok": int(row["rows_csv"]) == int(rows_loaded),
            }
        )
    load_audit_df = pd.DataFrame(load_rows)
load_audit_df

,table_name,rows_csv,rows_loaded,load_ok
0,public_dataset_modelo_proveedor_v1,16315,16315,True
1,public_dataset_modelo_proveedor_v2_candidates,155946,155946,True
2,public_dataset_modelo_proveedor_v2_excluded_ev...,911,911,True
3,public_dataset_modelo_proveedor_v3_context,155946,155946,True
4,public_dataset_modelo_v2_competition,155946,155946,True
5,public_dataset_modelo_v2_dispersion,155946,155946,True
6,public_dataset_modelo_v2_source_quality,155946,155946,True
7,public_dataset_modelo_v2_transport_only,155946,155946,True
8,public_dataset_modelo_v2_dispersion_plus_trans...,155946,155946,True
9,public_dataset_modelo_v2_transport_rebuilt_only,155946,155946,True


In [5]:
with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(VIEWS_SQL_PATH.read_text(encoding="utf-8"))
    checks = {
        "vw_inference_input_daily_rows": int(pd.read_sql_query("SELECT COUNT(*) AS rows FROM vw_inference_input_daily", conn).iloc[0]["rows"]),
        "vw_event_summary_daily_rows": int(pd.read_sql_query("SELECT COUNT(*) AS rows FROM vw_event_summary_daily", conn).iloc[0]["rows"]),
    }
    sample_daily = pd.read_sql_query("SELECT * FROM vw_inference_input_daily LIMIT 10", conn)
    sample_events = pd.read_sql_query("SELECT * FROM vw_event_summary_daily LIMIT 10", conn)

print(checks)
display(sample_daily)
display(sample_events)

{'vw_inference_input_daily_rows': 155946, 'vw_event_summary_daily_rows': 15404}


,event_id,fecha_evento,proveedor_candidato,producto_canonico,terminal_compra,coste_min_dia_proveedor,rank_coste_dia_producto,terminales_cubiertos,observaciones_oferta,candidatos_evento_count,coste_min_evento,coste_max_evento,spread_coste_evento,delta_vs_min_evento,ratio_vs_min_evento,litros_evento,dia_semana,mes,fin_mes
0,EVENT_8C109313E315,2021-04-03,SUPPLIER_020,PRODUCT_002,TERMINAL_001,795.34,1.0,4.0,4.0,13,795.34,814.53,19.19,0.00,1.000000,9999,0,4,0
1,EVENT_8C109313E315,2021-04-03,SUPPLIER_061,PRODUCT_002,TERMINAL_001,796.34,2.0,5.0,5.0,13,795.34,814.53,19.19,1.00,1.001257,9999,0,4,0
2,EVENT_8C109313E315,2021-04-03,SUPPLIER_001,PRODUCT_002,TERMINAL_001,796.62,3.0,1.0,1.0,13,795.34,814.53,19.19,1.28,1.001609,9999,0,4,0
3,EVENT_8C109313E315,2021-04-03,SUPPLIER_003,PRODUCT_002,TERMINAL_001,797.34,4.0,4.0,4.0,13,795.34,814.53,19.19,2.00,1.002515,9999,0,4,0
4,EVENT_8C109313E315,2021-04-03,SUPPLIER_013,PRODUCT_002,TERMINAL_001,798.74,5.0,4.0,4.0,13,795.34,814.53,19.19,3.40,1.004275,9999,0,4,0
5,EVENT_8C109313E315,2021-04-03,SUPPLIER_009,PRODUCT_002,TERMINAL_001,799.35,6.0,4.0,4.0,13,795.34,814.53,19.19,4.01,1.005042,9999,0,4,0
6,EVENT_8C109313E315,2021-04-03,SUPPLIER_023,PRODUCT_002,TERMINAL_001,800.45,7.0,1.0,1.0,13,795.34,814.53,19.19,5.11,1.006425,9999,0,4,0
7,EVENT_8C109313E315,2021-04-03,SUPPLIER_049,PRODUCT_002,TERMINAL_001,800.74,8.0,3.0,3.0,13,795.34,814.53,19.19,5.40,1.006790,9999,0,4,0
8,EVENT_8C109313E315,2021-04-03,SUPPLIER_054,PRODUCT_002,TERMINAL_001,801.74,9.0,3.0,3.0,13,795.34,814.53,19.19,6.40,1.008047,9999,0,4,0
9,EVENT_8C109313E315,2021-04-03,SUPPLIER_004,PRODUCT_002,TERMINAL_001,802.35,10.0,5.0,5.0,13,795.34,814.53,19.19,7.01,1.008814,9999,0,4,0


,event_id,candidatos_evento_count,coste_min_evento,coste_max_evento,spread_coste_evento
0,EVENT_000228BE85C3,12,897.16,915.345,18.185
1,EVENT_000C38E3FD8C,12,831.40,885.940,54.540
2,EVENT_00108405A516,13,846.34,921.340,75.000
3,EVENT_00115EE7F4D8,13,913.62,938.560,24.940
4,EVENT_001BA206EFF1,12,592.34,898.940,306.600
5,EVENT_001D060683E0,12,863.41,888.260,24.850
6,EVENT_001E84032733,8,688.37,717.590,29.220
7,EVENT_00221F503C1E,8,982.48,1067.340,84.860
8,EVENT_00228DA709E2,12,912.62,928.350,15.730
9,EVENT_002390A2EDB1,11,841.34,867.350,26.010


In [6]:
required_cols = set()
for column_group in contract["columns"].values():
    required_cols.update(column_group)
input_df = pd.read_csv(DATA_PUBLIC / "inference_inputs" / "example_real_day_2024-05-28.csv")
missing = required_cols - set(input_df.columns)
assert not missing, f"Faltan columnas de contrato en el input público: {sorted(missing)}"
print({"required_columns": len(required_cols), "input_shape": input_df.shape})

{'required_columns': 19, 'input_shape': (14, 29)}
